In [3]:
%load_ext autoreload
%autoreload 2

import numpy as np
import xarray as xr
import pandas as pd

import pyseaflux as sf
from pyseaflux import data

sf.data.utils.set_logger_level('INFO')

In [212]:
from voluptuous import All, Schema, Required, Optional, Any, Invalid
from functools import partial
from loguru import logger
import datetime
import yaml


def is_func(func_name):
    from pyseaflux.data import processors
    from pyseaflux.data import custom_funcs

    source_libs = (processors, custom_funcs)
    for library in source_libs:
        func = getattr(library, func_name, None)
        if func is not None:
            break
            
    if func is None:
        raise Invalid(
            f'Could not find the function `{func_name}` in '
            f'{str([sl.__name__ for sl in source_libs])}. '
            'Please edit `custom_funcs` to add the function'
        )
    return func

    
config_schema = Schema({
    'release': object,
    'atm_co2': dict,
    
    str: {
        Required('name'): str, 
        Optional('metadata'): dict,
        
        Required('urls'): [{
            Required('url'): str, 
            Optional('time'): {
                Required('start'): datetime.date, 
                Required('end'): datetime.date,
                Required('file_freq'): str},
            Optional(str): All([str])}],
        
        Required('fsspec_options'): {
            Required('cache_storage'): str,
            Optional('same_names'): bool,
            Optional('cache_mapper'): All(str, is_func),
            Optional(str): Any(str, dict)},
        
        Required('output_options'): {
            Required('output_storage'): str,
            Optional('output_freq'): str,
            Optional('delete_raw_files'): bool},
    
        Required('variables'): {str: str},
    
        Optional('processors'): [is_func]
        
}})


cat = data.utils.load_yaml_config('../configs/data_srouces_kryo.yaml')
schema = config_schema(cat)

In [237]:
def check_delete_raw_files(catalog):
    
    def warn_delete_raw_files(value):
        if value:
            raise Invalid(
                'Downloaded raw files for "{0}" will be deleted from '
                '"{1}" after each batch of final output is saved ')
    
    
    schema_catch_delete_raw_files = Schema({
        str: object,
        str: {
            Optional('output_options'): {
                str: object,
                Optional('delete_raw_files'): warn_delete_raw_files},
            str: object}
    })
    
    try:
        schema_catch_delete_raw_files(catalog)
    except Invalid as e:
        output_path = catalog[e.path[0]]['fsspec_options']['cache_storage']
        logger.warning(e.msg.format(e.path[0], output_path))

In [239]:
check_delete_raw_files(cat)

2025-03-27 18:20:38.172 | WARNING  | __main__:check_delete_raw_files:23 - Downloaded raw files for "era5" will be deleted from "../data/raw/era5/" after each batch of final output is saved 


In [103]:
getattr(processors, 'test', None)